# sEMG Prosthetic Gesture Classification
## Notebook 15: Final Publication Package & Scientific Reproducibility Bundle

**Project:** Machine Learning-Based sEMG Prosthetic Gesture Classification

---

### Research Objective & Scope
Assemble the submission-ready publication package by aggregating the **real, corrected**
outputs of Notebooks 10-14 (all of which were rewritten this project cycle to perform genuine
computation rather than read pre-existing, partly fabricated artifacts): master results
database, figure/table indexes, and corrected key-figure references for the manuscript.

**Provenance note:** the prior version of this notebook aggregated stale, pre-retuning and in
some cases fabricated figures -- e.g. 43.68% held-out accuracy (pre-308-trial-retuning),
a claimed "6-channel armband retains 98.6% of baseline performance" (real, measured value:
86.0%; see Notebook 13), a fabricated "Deployment Readiness Score: 96.5/100" (Notebook 14 only
ever computed a real 3-criterion pass/fail check, never a weighted /100 score), and a dataset
mislabel ("DB1" instead of the actual NinaPro DB2 used throughout). This rebuild reads the
real, current tables produced by the rewritten Notebooks 09-14 directly, so the numbers below
are reproducible by re-running this cell against the live `outputs/` directory.

In [1]:
import os, sys, json, time
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
outputs_dir = PROJECT_ROOT / 'outputs'
pub_pkg_dir = outputs_dir / 'publication_package'
manuscript_dir = outputs_dir / 'manuscript'
supplementary_dir = outputs_dir / 'supplementary'
indexes_dir = outputs_dir / 'indexes'
checklists_dir = outputs_dir / 'checklists'
metadata_dir = outputs_dir / 'metadata'
figures_dir = outputs_dir / 'figures'
tables_dir = outputs_dir / 'tables'
reports_dir = outputs_dir / 'reports'

for d in (pub_pkg_dir, manuscript_dir, supplementary_dir, indexes_dir, checklists_dir, metadata_dir):
    d.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)


Project Root: E:\Bio-Mechanics\semg-prosthetic-gesture-classification


## Section A: Real Master Results Database
Reads directly from the live, current tables produced by Notebooks 09-14 (all rewritten with
real computation this project cycle) rather than hardcoding figures.

In [2]:
# --- Dataset / feature engineering (structural facts, unchanged by retuning) ---
df_feat = pd.read_parquet(PROJECT_ROOT / "data/final/selected_features_top50.parquet")
n_subjects = df_feat["subject_id"].nunique()
n_windows = len(df_feat)
n_classes = df_feat["gesture_id"].nunique()

# --- Held-out test performance (Notebook 10, real retuned model) ---
overall = pd.read_csv(tables_dir / "overall_results.csv")
cb_heldout = overall[overall["Model"] == "CATBOOST"].iloc[0]

# --- Tuned hyperparameters (Notebook 09, real 308-trial Optuna search) ---
best_params = pd.read_csv(tables_dir / "best_parameters.csv")
best_params_cb = best_params[best_params["Model"] == "CATBOOST"].set_index("Parameter")
tuned_iterations = int(best_params_cb.loc["iterations", "Optimized Value"])
tuned_depth = int(best_params_cb.loc["depth", "Optimized Value"])

# --- LOSO cross-subject generalization (Notebook 11, real 40-fold Colab GPU run) ---
loso_stability = pd.read_csv(tables_dir / "stability_summary_v2.csv").set_index("Metric")
loso_acc_mean = loso_stability.loc["Accuracy", "Mean"]
loso_acc_std = loso_stability.loc["Accuracy", "Std Dev"]
loso_f1_mean = loso_stability.loc["Macro F1", "Mean"]
loso_f1_std = loso_stability.loc["Macro F1", "Std Dev"]

# --- Explainability (Notebook 12, real CatBoost-native SHAP) ---
global_ranking = pd.read_csv(tables_dir / "global_feature_ranking.csv")
channel_ranking = pd.read_csv(tables_dir / "channel_ranking.csv")
family_ranking = pd.read_csv(tables_dir / "feature_family_ranking.csv")
top_feature = global_ranking.iloc[0]
top_channel = channel_ranking.iloc[0]
top_family = family_ranking.iloc[0]

# --- Ablation studies (Notebook 13, real 20-config full-scale training) ---
deploy_rec = pd.read_csv(tables_dir / "deployment_recommendation_v2.csv")
best_channel_row = deploy_rec[deploy_rec["Deployment Profile"] == "Top 8 Channels"].iloc[0]

# --- Deployment (Notebook 14, real measured latency/memory) ---
latency = pd.read_csv(tables_dir / "latency_analysis.csv").set_index("Metric")["Value"]
memory = pd.read_csv(tables_dir / "memory_analysis.csv").set_index("Component")["Size_MB"]
deploy_metrics = pd.read_csv(tables_dir / "deployment_metrics.csv")
e2e_total_ms = deploy_metrics[deploy_metrics["Stage"] == "TOTAL"]["Latency_ms"].iloc[0]
deploy_checklist = pd.read_csv(tables_dir / "deployment_readiness_checklist.csv")
n_criteria_passed = int(deploy_checklist["Pass"].sum())
n_criteria_total = len(deploy_checklist)

print("All real source tables loaded successfully.")


All real source tables loaded successfully.


In [3]:
master_results_rows = [
    ("Dataset", "Total Subjects", n_subjects, "Subjects", "Notebook 02 (NinaPro DB2)"),
    ("Dataset", "Total Segmented Windows", n_windows, "Windows", "Notebook 04"),
    ("Dataset", "Gesture Classes", n_classes, "Classes", "Notebook 02"),
    ("Feature Engineering", "Total Extracted Features", 108, "Features", "Notebook 05 & 06"),
    ("Feature Selection", "Top Selected Features", 50, "Features", "Notebook 07"),
    ("Model Selection", "Best ML Architecture", "CatBoostClassifier", "Algorithm", "Notebook 08 & 09"),
    ("Hyperparameters", "Optuna Trials (Real, Completed)", 308, "Trials", "Notebook 09"),
    ("Hyperparameters", "Tuned Iterations", tuned_iterations, "Trees", "Notebook 09"),
    ("Hyperparameters", "Tuned Tree Depth", tuned_depth, "Depth", "Notebook 09"),
    ("Final Evaluation", "Held-Out Test Accuracy", round(cb_heldout["Accuracy"] * 100, 2), "%", "Notebook 10"),
    ("Final Evaluation", "Held-Out Test Macro F1", round(cb_heldout["Macro F1"] * 100, 2), "%", "Notebook 10"),
    ("LOSO Validation", "Cross-Subject Mean Accuracy", round(loso_acc_mean * 100, 2), "%% (+/- %.2f%%)" % (loso_acc_std * 100), "Notebook 11 (real 40-fold, Colab GPU)"),
    ("LOSO Validation", "Cross-Subject Mean Macro F1", round(loso_f1_mean * 100, 2), "%% (+/- %.2f%%)" % (loso_f1_std * 100), "Notebook 11 (real 40-fold, Colab GPU)"),
    ("Explainable AI", "Top Feature", top_feature["Feature"], "Mean Abs SHAP=%.3f" % top_feature["Mean_Absolute_SHAP"], "Notebook 12 (CatBoost-native SHAP)"),
    ("Explainable AI", "Top Channel Attribution", round(top_channel["Percentage (%)"], 2), "%% (%s)" % top_channel["Channel"], "Notebook 12"),
    ("Explainable AI", "Wavelet Domain Attribution", round(top_family["Percentage (%)"], 2), "%", "Notebook 12"),
    ("Ablation Studies", "Best Real Channel-Reduction Trade-off", "Top 8 Channels", "47/50 features, 99.5% F1 retention", "Notebook 13 (real 20-config Colab GPU sweep)"),
    ("Ablation Studies", "Top-8-Channel Macro F1 Retention", round(best_channel_row["Macro F1 Retention (%)"], 1), "%", "Notebook 13"),
    ("Deployment", "Native Single-Sample Latency", round(latency["Native single-sample latency (ms)"], 3), "ms", "Notebook 14"),
    ("Deployment", "ONNX Single-Sample Latency", round(latency["ONNX single-sample latency (ms)"], 3), "ms", "Notebook 14"),
    ("Deployment", "End-to-End Pipeline Latency (measured)", round(e2e_total_ms, 3), "ms", "Notebook 14 (real filtering+features+inference)"),
    ("Deployment", "Compressed Model Size (.cbm)", round(memory["CatBoost native binary (.cbm) on-disk/flash footprint"], 3), "MB", "Notebook 14"),
    ("Deployment", "ONNX Model Size (FP32)", round(memory["ONNX (FP32) on-disk footprint"], 3), "MB", "Notebook 14"),
    ("Deployment", "Deployment Readiness Criteria Passed", f"{n_criteria_passed}/{n_criteria_total}", "criteria (real pass/fail check, not a weighted score)", "Notebook 14"),
]
df_master_results = pd.DataFrame(master_results_rows, columns=["Category", "Metric", "Value", "Unit", "Source"])

df_master_results_parquet_safe = df_master_results.copy()
df_master_results_parquet_safe["Value"] = df_master_results_parquet_safe["Value"].astype(str)

for d in (metadata_dir, pub_pkg_dir):
    df_master_results.to_csv(d / "master_results.csv", index=False)
    df_master_results.to_json(d / "master_results.json", orient="records", indent=2)
    df_master_results_parquet_safe.to_parquet(d / "master_results.parquet", index=False)

display(df_master_results)


,Category,Metric,Value,Unit,Source
0,Dataset,Total Subjects,40,Subjects,Notebook 02 (NinaPro DB2)
1,Dataset,Total Segmented Windows,692276,Windows,Notebook 04
2,Dataset,Gesture Classes,50,Classes,Notebook 02
3,Feature Engineering,Total Extracted Features,108,Features,Notebook 05 & 06
4,Feature Selection,Top Selected Features,50,Features,Notebook 07
5,Model Selection,Best ML Architecture,CatBoostClassifier,Algorithm,Notebook 08 & 09
6,Hyperparameters,"Optuna Trials (Real, Completed)",308,Trials,Notebook 09
7,Hyperparameters,Tuned Iterations,393,Trees,Notebook 09
8,Hyperparameters,Tuned Tree Depth,5,Depth,Notebook 09
9,Final Evaluation,Held-Out Test Accuracy,45.42,%,Notebook 10


## Section B: Real Project Metadata

In [4]:
project_metadata = {
    "project_title": "Subject-Independent Surface Electromyography Pattern Recognition for Prosthetic Hand Control: A Multi-Domain Signal Processing and Explainability Framework",
    "dataset": "NinaPro DB2",
    "notebooks_completed": list(range(1, 16)),
    "primary_model": "CATBOOST",
    "optuna_trials_completed": 308,
    "test_accuracy_pct": round(cb_heldout["Accuracy"] * 100, 2),
    "test_macro_f1_pct": round(cb_heldout["Macro F1"] * 100, 2),
    "loso_mean_accuracy_pct": round(loso_acc_mean * 100, 2),
    "loso_mean_macro_f1_pct": round(loso_f1_mean * 100, 2),
    "loso_note": "Hyperparameter retuning improved held-out accuracy but did not meaningfully improve LOSO cross-subject generalization (see Notebook 11).",
    "best_channel_reduction_profile": "Top 8 Channels (47/50 features)",
    "best_channel_reduction_f1_retention_pct": round(best_channel_row["Macro F1 Retention (%)"], 1),
    "e2e_pipeline_latency_ms": round(e2e_total_ms, 3),
    "onnx_single_sample_latency_ms": round(latency["ONNX single-sample latency (ms)"], 3),
    "deployment_criteria_passed": f"{n_criteria_passed}/{n_criteria_total}",
    "reproducibility_status": "Notebooks 10-15 rewritten to perform real, executed computation (verified via nbconvert end-to-end execution); see individual notebook provenance notes for details.",
}
for d in (metadata_dir, pub_pkg_dir):
    with open(d / "project_metadata.json", "w", encoding="utf-8") as f:
        json.dump(project_metadata, f, indent=2)

print(json.dumps(project_metadata, indent=2))


{
  "project_title": "Subject-Independent Surface Electromyography Pattern Recognition for Prosthetic Hand Control: A Multi-Domain Signal Processing and Explainability Framework",
  "dataset": "NinaPro DB2",
  "notebooks_completed": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15
  ],
  "primary_model": "CATBOOST",
  "optuna_trials_completed": 308,
  "test_accuracy_pct": 45.42,
  "test_macro_f1_pct": 16.55,
  "loso_mean_accuracy_pct": 45.67,
  "loso_mean_macro_f1_pct": 16.43,
  "loso_note": "Hyperparameter retuning improved held-out accuracy but did not meaningfully improve LOSO cross-subject generalization (see Notebook 11).",
  "best_channel_reduction_profile": "Top 8 Channels (47/50 features)",
  "best_channel_reduction_f1_retention_pct": 99.5,
  "e2e_pipeline_latency_ms": 3.638,
  "onnx_single_sample_latency_ms": 0.121,
  "deployment_criteria_passed": "3/3",
  "reproducibility_status": "Notebooks 10-15 rewritten to per

## Section C: Master Figure & Table Indexes
Points to the real, current (`_v2` where applicable) figure and table files.

In [5]:
figure_index_rows = [
    ("Figure 11.1", "figure_11_01_complete_subject_ranking_v2.png", "Complete 40-Subject LOSO Macro F1 Ranking (real, retuned model).", "Notebook 11", "Results"),
    ("Figure 11.5", "figure_11_05_heldout_vs_loso_comparison_v2.png", "Held-Out Test vs 40-Fold LOSO Comparison.", "Notebook 11", "Results"),
    ("Figure 12.1", "figure_12_01_global_shap_summary.png", "Global SHAP Summary (CatBoost-native SHAP, real).", "Notebook 12", "Results"),
    ("Figure 12.4", "figure_12_04_channel_importance.png", "sEMG Channel SHAP Importance.", "Notebook 12", "Results"),
    ("Figure 12.5", "figure_12_05_feature_family_importance.png", "Feature Family (Domain) SHAP Importance.", "Notebook 12", "Results"),
    ("Figure 13.1", "figure_13_01_performance_drop_curves_v2.png", "Real Macro F1 vs Features Retained (SHAP removal + channel-reduction sweeps).", "Notebook 13", "Results"),
    ("Figure 13.2", "figure_13_02_feature_family_comparison_v2.png", "Real Feature Family Domain Comparison.", "Notebook 13", "Results"),
    ("Figure 14.4", "figure_14_04_e2e_pipeline_latency_breakdown.png", "Real End-to-End Pipeline Latency Breakdown.", "Notebook 14", "Results"),
    ("Figure 14.5", "figure_14_05_system_architecture_diagram.png", "Embedded Myoelectric Prosthetic System Architecture.", "Notebook 14", "Methodology"),
]
df_fig_idx = pd.DataFrame(figure_index_rows, columns=["Figure", "Filename", "Caption", "Notebook", "Suggested Section"])
for ext, writer in [("csv", df_fig_idx.to_csv), ("md", df_fig_idx.to_markdown)]:
    writer(indexes_dir / f"figure_index.{ext}", index=False)

table_index_rows = [
    ("Table 1", "overall_results.csv", "Held-Out Test Performance Comparison (CatBoost, XGBoost, LightGBM).", "Notebook 10", "Results"),
    ("Table 2", "subject_ranking_complete_v2.csv", "Complete 40-Subject LOSO Performance Ranking (real).", "Notebook 11", "Results"),
    ("Table 3", "stability_summary_v2.csv", "Real LOSO Fold Stability and Inter-Subject Variance.", "Notebook 11", "Results"),
    ("Table 4", "heldout_vs_loso_v2.csv", "Real Held-Out vs LOSO Comparison.", "Notebook 11", "Results"),
    ("Table 5", "global_feature_ranking.csv", "Real SHAP Global Feature Ranking.", "Notebook 12", "Results"),
    ("Table 6", "channel_ranking.csv", "Real SHAP Channel Importance Ranking.", "Notebook 12", "Results"),
    ("Table 7", "feature_removal_summary_v2.csv", "Real Feature Removal Ablation (Top/Bottom SHAP).", "Notebook 13", "Results"),
    ("Table 8", "channel_efficiency_v2.csv", "Real Channel Reduction Performance/Latency Trade-off.", "Notebook 13", "Results"),
    ("Table 9", "deployment_recommendation_v2.csv", "Real Deployment Profile Recommendation.", "Notebook 13", "Results"),
    ("Table 10", "deployment_readiness_checklist.csv", "Real Deployment Readiness Pass/Fail Criteria.", "Notebook 14", "Results"),
]
df_tbl_idx = pd.DataFrame(table_index_rows, columns=["Table", "Filename", "Caption", "Notebook", "Suggested Section"])
for ext, writer in [("csv", df_tbl_idx.to_csv), ("md", df_tbl_idx.to_markdown)]:
    writer(indexes_dir / f"table_index.{ext}", index=False)

print("Figure index:"); display(df_fig_idx)
print("Table index:"); display(df_tbl_idx)


Figure index:


,Figure,Filename,Caption,Notebook,Suggested Section
0,Figure 11.1,figure_11_01_complete_subject_ranking_v2.png,Complete 40-Subject LOSO Macro F1 Ranking (rea...,Notebook 11,Results
1,Figure 11.5,figure_11_05_heldout_vs_loso_comparison_v2.png,Held-Out Test vs 40-Fold LOSO Comparison.,Notebook 11,Results
2,Figure 12.1,figure_12_01_global_shap_summary.png,"Global SHAP Summary (CatBoost-native SHAP, real).",Notebook 12,Results
3,Figure 12.4,figure_12_04_channel_importance.png,sEMG Channel SHAP Importance.,Notebook 12,Results
4,Figure 12.5,figure_12_05_feature_family_importance.png,Feature Family (Domain) SHAP Importance.,Notebook 12,Results
5,Figure 13.1,figure_13_01_performance_drop_curves_v2.png,Real Macro F1 vs Features Retained (SHAP remov...,Notebook 13,Results
6,Figure 13.2,figure_13_02_feature_family_comparison_v2.png,Real Feature Family Domain Comparison.,Notebook 13,Results
7,Figure 14.4,figure_14_04_e2e_pipeline_latency_breakdown.png,Real End-to-End Pipeline Latency Breakdown.,Notebook 14,Results
8,Figure 14.5,figure_14_05_system_architecture_diagram.png,Embedded Myoelectric Prosthetic System Archite...,Notebook 14,Methodology


Table index:


,Table,Filename,Caption,Notebook,Suggested Section
0,Table 1,overall_results.csv,Held-Out Test Performance Comparison (CatBoost...,Notebook 10,Results
1,Table 2,subject_ranking_complete_v2.csv,Complete 40-Subject LOSO Performance Ranking (...,Notebook 11,Results
2,Table 3,stability_summary_v2.csv,Real LOSO Fold Stability and Inter-Subject Var...,Notebook 11,Results
3,Table 4,heldout_vs_loso_v2.csv,Real Held-Out vs LOSO Comparison.,Notebook 11,Results
4,Table 5,global_feature_ranking.csv,Real SHAP Global Feature Ranking.,Notebook 12,Results
5,Table 6,channel_ranking.csv,Real SHAP Channel Importance Ranking.,Notebook 12,Results
6,Table 7,feature_removal_summary_v2.csv,Real Feature Removal Ablation (Top/Bottom SHAP).,Notebook 13,Results
7,Table 8,channel_efficiency_v2.csv,Real Channel Reduction Performance/Latency Tra...,Notebook 13,Results
8,Table 9,deployment_recommendation_v2.csv,Real Deployment Profile Recommendation.,Notebook 13,Results
9,Table 10,deployment_readiness_checklist.csv,Real Deployment Readiness Pass/Fail Criteria.,Notebook 14,Results


## Section D: Correcting Stale Numbers in Manuscript Auxiliary Sections
The 18 standalone manuscript markdown files (`outputs/manuscript/`) contain narrative prose
written before this project's Optuna retuning and NB10-14 rewrites. Rather than a full prose
regeneration (which would duplicate the separately-maintained BSPC manuscript), this section
performs a targeted, real-numbers correction pass: replacing every stale figure identified by
grep against the corrected source tables, and fixing the "DB1"/"DB5" dataset mislabel to the
actual NinaPro DB2 used throughout this project.

In [6]:
held_acc_str = f"{cb_heldout['Accuracy'] * 100:.2f}%"
held_f1_str = f"{cb_heldout['Macro F1'] * 100:.2f}%"
loso_f1_str = f"{loso_f1_mean * 100:.2f}% +/- {loso_f1_std * 100:.2f}% (real 40-fold LOSO, Colab GPU)"
top_channel_pct_str = f"{top_channel['Percentage (%)']:.2f}%"
top_family_pct_str = f"{top_family['Percentage (%)']:.2f}%"
onnx_lat_str = f"{latency['ONNX single-sample latency (ms)']:.3f} ms"
e2e_lat_str = f"{e2e_total_ms:.3f} ms"
margin_str = f"{100 * (1 - e2e_total_ms / 50):.1f}%"
criteria_str = f"{n_criteria_passed}/{n_criteria_total}"
channel_family_str = f"{top_channel['Channel']} ({top_channel_pct_str}) and {top_family['Feature_Family']} features ({top_family_pct_str})"

replacements = {
    "43.68% Accuracy and 15.68% Macro F1": f"{held_acc_str} Accuracy and {held_f1_str} Macro F1",
    "43.68%": held_acc_str,
    "15.68%": held_f1_str,
    "[15.22%, 16.14%] 95% CI": loso_f1_str,
    "Channel 11 (40.07%) and Wavelet Domain features (69.79%)": channel_family_str,
    "40.07%": top_channel_pct_str,
    "69.79%": top_family_pct_str,
    "6-channel armband retains 98.6% of baseline Macro F1": "top-8-channel configuration retains 99.5% of baseline Macro F1 (real, measured; a 6-channel configuration retains 86.0%)",
    "98.6%": "99.5% (top-8-channel; 6-channel real retention is 86.0%)",
    "reducing inference latency to 1.12 ms (single-sample) and total end-to-end pipeline latency to 6.40 ms (87.2% safety margin below 50 ms budget)":
        f"real ONNX single-sample latency of {onnx_lat_str} and a real measured end-to-end pipeline latency of {e2e_lat_str}, comfortably within the 50 ms budget",
    "1.12 ms": onnx_lat_str,
    "6.40 ms": e2e_lat_str,
    "6.4 ms": e2e_lat_str,
    "87.2%": margin_str,
    "Deployment Readiness Score of 96.5 / 100": f"real deployment readiness check of {criteria_str} criteria passed",
    "96.5": f"{criteria_str} criteria",
    "benchmark DB1 dataset": "NinaPro DB2 benchmark dataset",
    "(DB1)": "(NinaPro DB2)",
    "DB1": "DB2",
}

corrected_files = []
for md_file in list(manuscript_dir.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    original = text
    for old, new in replacements.items():
        text = text.replace(old, new)
    if text != original:
        md_file.write_text(text, encoding="utf-8")
        corrected_files.append(md_file.name)

for pkg_file in (pub_pkg_dir / "project_summary.md",):
    if pkg_file.exists():
        text = pkg_file.read_text(encoding="utf-8")
        original = text
        for old, new in replacements.items():
            text = text.replace(old, new)
        if text != original:
            pkg_file.write_text(text, encoding="utf-8")
            corrected_files.append(pkg_file.name)

print(f"Corrected {len(corrected_files)} files with stale/fabricated figures:")
for f in corrected_files:
    print(" -", f)


Corrected 7 files with stale/fabricated figures:
 - 02_abstract.md
 - 08_results.md
 - 10_conclusion.md
 - 16_data_availability.md
 - figure_captions.md
 - table_captions.md
 - project_summary.md


## Section E: Completeness Report (Real)

In [7]:
completeness_lines = [
    "# Project Completeness & Verification Report: Notebooks 01-15",
    "",
    "**Regenerated:** " + time.strftime("%Y-%m-%d"),
    "**Status:** Notebooks 10-15 rewritten this project cycle to perform real, executed",
    "computation (verified via `jupyter nbconvert --execute` end-to-end for each).",
    "",
    "### Verified Real-Computation Notebooks:",
    "- **Notebook 09 Hyperparameter Tuning**: real 308-trial Optuna search (Colab, resumable)",
    "- **Notebook 10 Final Model Evaluation**: real held-out inference, McNemar tests, calibration",
    "- **Notebook 11 LOSO Validation**: real 40-fold training (Colab T4 GPU, 54.7 min)",
    "- **Notebook 12 Explainable AI**: real CatBoost-native SHAP (TreeExplainer segfault fixed)",
    "- **Notebook 13 Ablation Studies**: real 20-configuration full-scale training (Colab T4 GPU)",
    "- **Notebook 14 Deployment & Optimization**: real latency/memory/quantization measurement",
    "- **Notebook 15 Publication Package**: real aggregation of the above (this notebook)",
]
(checklists_dir / "completeness_report.md").write_text("\n".join(completeness_lines), encoding="utf-8")
print("\n".join(completeness_lines))


# Project Completeness & Verification Report: Notebooks 01-15

**Regenerated:** 2026-08-03
**Status:** Notebooks 10-15 rewritten this project cycle to perform real, executed
computation (verified via `jupyter nbconvert --execute` end-to-end for each).

### Verified Real-Computation Notebooks:
- **Notebook 09 Hyperparameter Tuning**: real 308-trial Optuna search (Colab, resumable)
- **Notebook 10 Final Model Evaluation**: real held-out inference, McNemar tests, calibration
- **Notebook 11 LOSO Validation**: real 40-fold training (Colab T4 GPU, 54.7 min)
- **Notebook 12 Explainable AI**: real CatBoost-native SHAP (TreeExplainer segfault fixed)
- **Notebook 13 Ablation Studies**: real 20-configuration full-scale training (Colab T4 GPU)
- **Notebook 14 Deployment & Optimization**: real latency/memory/quantization measurement
- **Notebook 15 Publication Package**: real aggregation of the above (this notebook)


## Notebook Summary

### Executive Summary
This notebook assembled the publication package by aggregating the **real, corrected**
outputs of the rewritten Notebooks 09-14, replacing the prior version's stale (pre-retuning)
and in some cases fabricated figures.

### Corrected Findings (real vs. prior stale/fabricated values)
| Metric | Prior (stale) | Real (this rebuild) |
|---|---|---|
| Held-out Accuracy | 43.68% | ~45.4% |
| Held-out Macro F1 | 15.68% | ~16.5% |
| LOSO Macro F1 | 15.68% (duplicated held-out figure) | ~16.4% (real 40-fold) |
| Best channel-reduction retention | "6-channel: 98.6%" | Top-8-channel: 99.5%; 6-channel: 86.0% |
| Deployment readiness | "96.5 / 100" (fabricated score) | 3/3 real pass/fail criteria |
| Dataset label | "DB1" (mislabeled) | NinaPro DB2 (correct) |

### Publication Artifacts Regenerated
- **Master Results Database**: `outputs/metadata/` and `outputs/publication_package/`
  (`master_results.csv/json/parquet`, `project_metadata.json`)
- **Master Indexes**: `outputs/indexes/figure_index.*`, `table_index.*`
- **Manuscript Corrections**: targeted numeric/label corrections applied across the 18
  standalone manuscript markdown files in `outputs/manuscript/`
- **Completeness Report**: `outputs/checklists/completeness_report.md`

### Remaining Work
The BSPC journal manuscript (`publication/journals/BSPC/submission_files/02_Manuscript.docx`)
was drafted before this project cycle's Optuna retuning and NB10-15 rewrites, and still
contains the pre-correction numbers throughout. It needs a dedicated pass to incorporate the
real, current figures established across this project's rewritten notebooks.